# Zero-Shot Prompting

Issue direct task instructions with no input-output examples — ideal when the task is familiar and the model can infer format from clear constraints.


## 1. Overview

This guide covers:

- Zero-shot: instructions only, zero exemplars
- When zero-shot is sufficient vs when to add examples
- A quick local rubric for judging zero-shot prompt quality
- Zero-shot classification and structured extraction with LangChain


## 2. Motivation

Zero-shot is the default: fastest to write, fewest tokens, no example curation. Modern chat models handle many tasks (summarize, classify, translate) from a well-written instruction alone.

Reach for one-shot or few-shot when format drifts, labels are domain-specific, or evaluations show systematic errors.


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **Zero-shot** | Prompt with task description but no input-output examples |
| **Instruction following** | Model complies from natural-language rules |
| **Label space** | Allowed categories or fields you define in the prompt |
| **Rubric** | Criteria to score output quality without golden examples |
| **Structured output** | Pydantic schema enforced via `with_structured_output` |

### 3.2 How it works

The model relies on weights trained on broad internet text plus your explicit instructions. Clear label definitions and output format reduce ambiguity without showing examples.

### 3.3 When to use zero-shot

**Use when:** task is common, labels are standard, format is simple (JSON enum, yes/no, short summary).

**Avoid when:** niche taxonomy, strict schema, or repeated format errors — add examples (one-shot / few-shot notebooks).

**Trade-offs:** lowest token cost and maintenance; less control than few-shot for edge cases.


## 4. Architecture

Zero-shot is a **single forward pass**: instructions + new input go in; a constrained answer comes out. No exemplars, no tool loop.

### 4.1 End-to-end data flow

```mermaid
flowchart TB
    subgraph setup["Notebook setup"]
        env[".env<br/>OPENAI_API_KEY"]
        llm["ChatOpenAI"]
        env --> llm
    end

    subgraph prompt_layer["Prompt layer — no examples"]
        sys["System message<br/>policy + output rules"]
        human["Human message<br/>task · labels · constraints<br/>+ &lt;input&gt;…&lt;/input&gt;"]
        tmpl["ChatPromptTemplate"]
        sys --> tmpl
        human --> tmpl
    end

    subgraph chain["LCEL chain"]
        branch{Structured?}
        text["Free-text label<br/>+ assert in code"]
        schema["with_structured_output<br/>Pydantic model"]
        tmpl --> llm
        llm --> branch
        branch -->|classify| text
        branch -->|extract| schema
    end

    text --> out["Validated result"]
    schema --> out
```

### 4.2 Message shape (what the model actually sees)

```text
┌─ system ─────────────────────────────────────────┐
│  Role / policy (e.g. "use only allowed labels")  │
└──────────────────────────────────────────────────┘
┌─ human ──────────────────────────────────────────┐
│  Task: …                                         │
│  Allowed labels / fields: …                      │
│  Constraints: …                                  │
│  Output format: …                                │
│  Input:                                          │
│  <input>                                         │
│  …untrusted user text…                           │
│  </input>                                        │
│                                                  │
│  ← no Example / Input / Output blocks here       │
└──────────────────────────────────────────────────┘
```

### 4.3 Chain recipes used in this notebook

| Path | LCEL shape | When to use |
|------|------------|-------------|
| **Classification** | `ChatPromptTemplate \| llm` → strip → `assert label in allowed` | Small closed label set |
| **Extraction** | `ChatPromptTemplate \| llm.with_structured_output(Schema)` | Multi-field JSON / enums |

```text
prompt | llm                                # label text
prompt | llm.with_structured_output(Model)  # TicketExtract, etc.
```

This notebook is self-contained: setup, prompts, and chains all live in the cells below.


## 5. Local Python Examples


In [1]:
# Zero-shot prompt + simple rubric — no API required
task = "Classify customer feedback sentiment."
labels = ["positive", "neutral", "negative"]
constraints = "If mixed sentiment, choose neutral."
output_format = "Reply with the label only (one word, lowercase)."
sample = "Delivery was fast but the packaging was damaged."

prompt = f"""Task: {task}
Allowed labels: {', '.join(labels)}
Constraints: {constraints}
Output format: {output_format}
Input:
<input>
{sample.strip()}
</input>"""

print(prompt)

issues = []
if not task.strip():
    issues.append("missing task")
if len(labels) < 2:
    issues.append("need at least two labels for classification")
if "only" not in output_format.lower():
    issues.append("output format should say 'only' to reduce preamble")

print("rubric issues:", issues or "none")


Task: Classify customer feedback sentiment.
Allowed labels: positive, neutral, negative
Constraints: If mixed sentiment, choose neutral.
Output format: Reply with the label only (one word, lowercase).
Input:
<input>
Delivery was fast but the packaging was damaged.
</input>
rubric issues: none


## 6. LangChain Examples

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv(root / ".env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
```


In [2]:
# Setup: load .env and create ChatOpenAI (standalone — no project helpers)
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / ".env").is_file() or (p / "requirements.txt").is_file()
)
load_dotenv(root / ".env")

api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key.strip() or "your_openai_api_key" in api_key.lower():
    raise SystemExit("Set OPENAI_API_KEY in .env before running the API cells.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0)
print("Model ready:", llm.model_name)


Model ready: gpt-4o-mini


In [3]:
# Zero-shot classification — template vars + post-validate label set
labels = ["positive", "neutral", "negative"]

classify_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You classify text using only the allowed labels."),
        (
            "human",
            "Task: {task}\n"
            "Allowed labels: {labels}\n"
            "Constraints: {constraints}\n"
            "Output format: Reply with the label only (one word, lowercase).\n"
            "Input:\n<input>\n{text}\n</input>",
        ),
    ]
)

classify_chain = classify_prompt | llm
result = classify_chain.invoke(
    {
        "task": "Classify customer feedback sentiment.",
        "labels": ", ".join(labels),
        "constraints": "If mixed sentiment, choose neutral.",
        "text": "Love the new dashboard — loads in under a second now!",
    }
)

label = result.content.strip().lower()
assert label in labels, f"unexpected label: {label!r}"
print("Classification:", label)


Classification: positive


In [4]:
# Zero-shot extraction — Pydantic schema via with_structured_output
class TicketExtract(BaseModel):
    product: str = Field(description="Product or surface affected")
    issue: str = Field(description="Short description of the problem")
    urgency: Literal["low", "medium", "high"]


extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Extract structured fields from support tickets. "
            "Use only the schema fields: product, issue, urgency.",
        ),
        ("human", "Ticket:\n<input>\n{ticket}\n</input>"),
    ]
)

extract_llm = ChatOpenAI(model=MODEL, temperature=0.1).with_structured_output(TicketExtract)
extract_chain = extract_prompt | extract_llm

ticket = extract_chain.invoke(
    {
        "ticket": (
            "User cannot reset password; error 500 since 09:00 UTC. "
            "Billing portal affected."
        )
    }
)

print("Parsed model:", ticket)
print("As dict:", ticket.model_dump())


Parsed model: product='Billing portal' issue='User cannot reset password; error 500 since 09:00 UTC' urgency='high'
As dict: {'product': 'Billing portal', 'issue': 'User cannot reset password; error 500 since 09:00 UTC', 'urgency': 'high'}


## 7. Implementation notes

1. **`load_dotenv` + `ChatOpenAI`** — Load the key from `.env`, then build the model in the notebook.
2. **`ChatPromptTemplate` vars** — Keep `{task}` / `{labels}` / `{text}` so one chain serves many inputs.
3. **Temperature 0** — Prefer for classification; use a separate `ChatOpenAI(..., temperature=0.1)` when needed.
4. **Validate labels in code** — `assert label in labels` (or structured output for enums).
5. **`with_structured_output`** — Prefer over regex/JSON string parsing for extraction.
6. **Reuse the chain** — Build `prompt | llm` once; call `.invoke` / `.batch` per row.


## 8. Best practices

- Name **allowed labels** explicitly; define tie-break rules ("if unsure, neutral").
- Put **definitions** for ambiguous classes in the prompt.
- Delimit **untrusted input** with XML tags such as `<input>...</input>`.
- Prefer **Pydantic + `with_structured_output`** for multi-field extraction.
- Validate outputs in code (enum check, schema) — do not trust free text alone.
- Keep zero-shot prompts short; every sentence should change model behavior.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Label plus explanation | Weak output format | "Reply with the label only" or structured enum |
| Invented category | Label not in allowed set | List labels; assert / Literal in schema |
| JSON with prose wrapper | Free-text JSON ask | Use `with_structured_output` |
| Inconsistent urgency | Subjective scale | `Literal["low","medium","high"]` in schema |
| Works in demo, fails in prod | Different input distribution | Evaluate on real samples; add shots |
| High token use | Long policy in every call | Move stable rules to the `system` message |


## 10. Validation checklist

1. Run the local rubric cell; confirm zero issues for a complete prompt.
2. Run classification; output is one allowed label (`assert` passes).
3. Run extraction; result is a `TicketExtract` instance with valid `urgency`.
4. Confirm `OPENAI_API_KEY` loads from `.env`.
5. Compare temperature 0 vs 0.7 on classification — confirm 0 is more stable.


## 11. Summary

- Zero-shot = clear instructions, **no examples**.
- Best for standard tasks with explicit label spaces and formats.
- Use LangChain as `ChatPromptTemplate | llm`, with structured output for schemas.
- This notebook stands alone — no shared `assets` imports required.
- Upgrade to one-shot/few-shot when evals fail.

**Next:** `One_Shot_Prompting.ipynb` — steer format with a single exemplar.
